[Home](../../README.md)

### Feature Engineering

This Jupyter Notepad is a selection of data engineering processes you can apply to your data before model training to maximise the performance of your machine learning model. For this demonstration we will engineer new or improved features for the diabetes data you previously wrangled.

#### Feature Engineering Process
- Deriving new variables from existing ones
    - Encoding categorical features
    - Calculating new features from existing features
- Combining features/feature interactions
- Identifying the most relevant features for the model
- Transforming Features
  - [Dividing Data into categories](https://web.ma.utexas.edu/users/mks/statmistakes/dividingcontinuousintocategories.html)
  - Mathematical transformations (for example logarithmic transformations). Logarithmic transformations are a powerful tool in the world of statistical analysis. They are often used to transform data that exhibit skewness or other irregularities, making it easier to analyze, visualize, and interpret the results.
- Creating Domain-Specific Features that incorporating knowledge from the specific domain to create features that capture important characteristics of the data.

#### Load the required dependencies

In [101]:
# Import frameworks
import pandas as pd
from datetime import date

####  Store the data as a local variable

The data frame is a Pandas object that structures your tabular data into an appropriate format. It loads the complete data in memory so it is now ready for preprocessing.

In [102]:
results = pd.read_csv("2.2.1.wrangled_results.csv")
results_all_events = pd.read_csv("2.2.1.unwrangled_results.csv")
persons = pd.read_csv("2.2.1.wrangled_persons.csv")
countries = pd.read_csv("2.2.1.wrangled_countries.csv")
competitions = pd.read_csv("2.2.1.wrangled_competitions.csv")
current_date = date.today().year

#### Calculating Competitions entered

In [103]:
competitions_entered = results_all_events.groupby("person_id")["competition_id"].nunique().reset_index()
competitions_entered.columns = ["person_id", "competitions_entered"]
print(competitions_entered.head())

    person_id  competitions_entered
0  1982BORS01                     1
1  1982BRIN01                     1
2  1982CHIL01                     1
3  1982FRID01                     4
4  1982GALR01                     1


#### Calculating Years Active


In [104]:
results["competition_id"] = results["competition_id"].astype(str)
competitions["id"] = competitions["id"].astype(str)

year_results = results.merge(competitions[["id", "year"]], left_on="competition_id", right_on="id", how="left")
years_active = (year_results.groupby("person_id")["year"].agg(["min", "max"]).reset_index())

years_active["years_active"] = current_date - years_active["min"]
years_active = years_active[["person_id", "years_active"]]
print(years_active.head())

    person_id  years_active
0  1982FRID01            23
1  1982PETR01            22
2  1982RAZO01            22
3  1982VALD01            14
4  2003AKIM01            21


#### Person Country

In [105]:
person_country = year_results.merge(competitions[["id", "country_id"]], left_on="competition_id", right_on="id", how="left")
person_country = person_country.groupby("person_id")["country_id"].agg(lambda avg: avg.mode()[0]).reset_index()
person_country.columns = ["person_id", "country"]
print(person_country.head())

    person_id      country
0  1982FRID01          USA
1  1982PETR01          USA
2  1982RAZO01  Netherlands
3  1982VALD01         Peru
4  2003AKIM01        Japan


#### Person Continent

In [106]:
person_continent = person_country.merge(countries[["id", "continent_id"]], left_on="country", right_on="id", how="left")
person_continent = person_continent[["person_id", "continent_id"]]
person_continent.columns = ["person_id", "continent_id"]
print(person_continent.head())

    person_id    continent_id
0  1982FRID01  _North America
1  1982PETR01  _North America
2  1982RAZO01         _Europe
3  1982VALD01  _South America
4  2003AKIM01           _Asia


## Combining features/feature interactions

#### Calculating Competitions Per Year


In [107]:
competitions_per_year = competitions_entered.merge(years_active, on="person_id", how="left")
competitions_per_year["competitions_per_year"] = round(competitions_per_year["competitions_entered"] / competitions_per_year["years_active"].replace(0,1), 2)
competitions_per_year = competitions_per_year.dropna(subset=["years_active", "competitions_per_year"])
print(competitions_per_year.head())

     person_id  competitions_entered  years_active  competitions_per_year
3   1982FRID01                     4          23.0                   0.17
8   1982PETR01                    32          22.0                   1.45
9   1982RAZO01                    50          22.0                   2.27
18  1982VALD01                     2          14.0                   0.14
19  2003AKIM01                    19          21.0                   0.90


## Domain specific features

Calculating the number of events entered counts more than just the perticipants 3x3x3 entries, however it is known that many solving algorithms are similar if not exactly the same, no matter the cube. Hence, having experience in other solving categories, before or after having a 3x3x3 attempt, significantly increases the perticipants experience and would therefore decrease their solve time.  

#### Calculating Events Completed

In [108]:
events_completed = results_all_events.groupby("person_id")["event_id"].count().reset_index()
events_completed = events_completed[events_completed["person_id"].isin(results["person_id"])]
events_completed.columns = ["person_id", "events_completed"]
print(events_completed.head())

     person_id  events_completed
3   1982FRID01                 9
8   1982PETR01                91
9   1982RAZO01                95
18  1982VALD01                 2
19  2003AKIM01                58


#### Calculating Events Per Competition  

In [109]:
events_per_competition = competitions_entered.merge(events_completed, on="person_id", how="left")
events_per_competition["events_per_competition"] = round(events_per_competition["events_completed"] / events_per_competition["competitions_entered"].replace(0,1), 2)
events_per_competition = events_per_competition.dropna(subset=["events_completed", "events_per_competition"])
print(events_per_competition.head())

     person_id  competitions_entered  events_completed  events_per_competition
3   1982FRID01                     4               9.0                    2.25
8   1982PETR01                    32              91.0                    2.84
9   1982RAZO01                    50              95.0                    1.90
18  1982VALD01                     2               2.0                    1.00
19  2003AKIM01                    19              58.0                    3.05


# MERGE


In [110]:
data_frame = results.copy()

data_frame = data_frame.merge(competitions_entered, on="person_id", how="left")
data_frame = data_frame.merge(years_active, on="person_id", how="left")
data_frame = data_frame.merge(events_completed, on="person_id", how="left")
data_frame = data_frame.merge(competitions_per_year[["person_id", "competitions_per_year"]], on="person_id", how="left")
data_frame = data_frame.merge(person_country, on="person_id", how="left")
data_frame = data_frame.merge(person_continent, on="person_id", how="left")
data_frame = data_frame.merge(events_per_competition[["person_id", "events_per_competition"]], on="person_id", how="left")
data_frame = data_frame.drop(columns=["pos", "best", "competition_id", "event_id", "person_name", "person_country_id"])

print(data_frame.head())

   average   person_id  competitions_entered  years_active  events_completed  \
0     2128  2007AMAN01                     2            19                 8   
1     2140  2004ROUA01                     6            22                14   
2     2637  2005SIMO01                    28            21               175   
3     2637  2007MALL01                     6            19                33   
4     2906  2007DESM01                    10            19                29   

   competitions_per_year country continent_id  events_per_competition  
0                   0.11  France      _Europe                    4.00  
1                   0.27  France      _Europe                    2.33  
2                   1.33  France      _Europe                    6.25  
3                   0.32  France      _Europe                    5.50  
4                   0.53  France      _Europe                    2.90  


####  Deriving new variables from existing ones

##### Encoding categorical variables

Data Encoding converts textual data into numerical format, so that it can be used as input for algorithms to process. The reason for encoding is that most machine learning algorithms work with numbers and not with text or categorical variables.

To encode the 'SEX' column you will assigning a number value to the gender. Because the data set only provides 2 values we will use -1 and 1.

In [111]:
# data_frame['SEX'] = data_frame['SEX'].apply(lambda gender: -1 if gender.lower() == 'male' else 1 if gender.lower() == 'female' else None)
# print(data_frame['SEX'].head())

##### Calculating Age

In the context of medical diagnosis of a lifestyle disease a persons date of birth has limited influence on the target. However, their age is highly relevant. So we will convert two dates into a age. You could consider further encoding this into age brackets or scalling the data, don't forget to use [2.1.2.data.records.md](../2.1.Data_Wrangling/2.1.2.data.records.md) to record any scalling or encoding.

In [112]:
# Convert the 'DoB' and 'DoTest' columns to datetime
# data_frame['DoB'] = pd.to_datetime(data_frame['DoB'], format='%d/%m/%Y')
# data_frame['DoT'] = pd.to_datetime(data_frame['DoT'], format='%d/%m/%Y')

# # Calculate the year difference
# data_frame['Age'] = ((data_frame['DoT'] - data_frame['DoB']).dt.days  / 365.25).round()

# # Print the result
# print(data_frame[['DoB', 'DoT', 'Age']])

#### Combining features/feature interactions

While individual features can be powerful predictors, their interactions often carry even more information. Feature interaction engineering is the process of creating new features that represent the interaction between two or more features.

In this, case some domain knowledge and data analysis have informed you that the BMI and AGE are risk multipliers (the greater the age and the greater the BMI the greater the feature). From this we can  risk value based on the feature interactions.

In [113]:
# Calculate the year difference and round to an integer
# data_frame['Age'] = ((data_frame['DoT'] - data_frame['DoB']).dt.days / 365.25).round().astype(int)

# # Create the 'Risk' column
# data_frame['Risk'] = data_frame['BMI'] * data_frame['Age']

# # Calculate the percentage of the maximum risk
# data_frame['Risk%'] = (data_frame['Risk'] / data_frame['Risk'].max()).round(2)

# # Print the result
# print(data_frame[['Age', 'BMI', 'Risk%']])

#### Transforming Features

Filtering is like applying the where clause in a database. It is widely used and can help when you need to work on a specific subset of your data. For our use case, let us filter the data to only include rows where the 'SEX' is 'Male'. There is no method call for this, we can just use conditional indexing to fulfil our purpose.

In this, case some domain knowledge and data analysis have informed you that there is 'bimodality' in the data and males and females have a different trends. 

In [114]:
# # Filter the data to -1 only
# data_frame = data_frame[data_frame['SEX'] == -1]

# # Print the result
# print(data_frame[['Age', 'SEX', 'Target']])

#### Creating Domain-Specific Features

Domain knowledge is about understanding the domain or subject area of the data. In This case the domain is 'health' and more specifically   'Epidemiology' which is the study of how often diseases occur in different groups of people and why.

The column called '1st Degree Relatives' is a domain specific feature as is records the number of family members in the individuals direct bloodline who have developed type 2 adult onset diabetes. Domain specific knowledge, is that Family history of disease in first degree relatives is a major risk factor, especially for premature events.

First we will convert convert the FDR value to a risk percentage, because the risk can never be 0 (will never happen) or 100% (will definitely happen) we will scale the result between 0.15 and 0.85.

In [115]:
# Calculate the family history risk
# data_frame['FHRisk'] = (data_frame['FDR'] / data_frame['FDR'].max())

# # Scale the result between 0.15 and 0.85
# min_val = 0.15
# max_val = 0.85
# data_frame['FHRisk'] = (((data_frame['FHRisk'] - data_frame['FHRisk'].min()) / (data_frame['FHRisk'].max() - data_frame['FHRisk'].min())) * (max_val - min_val) + min_val).round(2)

# # Print the result
# print(data_frame[['Age', 'FDR', 'FHRisk']])

Then to make it even more meaningful, we will combine it with the `Risk` feature we engineered using the `AGE` and `BMI` features to create a combined risk 'interaction feature' that captures real-world relationships between the features.

Again we will scale the result between 0.15 and 0.85.

In [116]:
# data_frame['CombRisk'] = (data_frame['FHRisk'] * data_frame['Risk%']).round(2)

# min_val = 0.15
# max_val = 0.85
# data_frame['CombRisk'] = (((data_frame['CombRisk'] - data_frame['CombRisk'].min()) / (data_frame['CombRisk'].max() - data_frame['CombRisk'].min())) * (max_val - min_val) + min_val).round(2)

# # Print the result
# print(data_frame[['Age', 'Risk%', 'FHRisk', 'CombRisk']])

#### Save the wrangled and engineered data to CSV

In [117]:
data_frame.to_csv('../2.3.Model_Training/2.3.1.model_ready_data.csv', index=False)